<!--
Copyright (c) Meta Platforms, Inc. and affiliates.
All rights reserved.

This source code is licensed under the CC-BY-NC-4.0 license found in the
LICENSE file in the root directory of this source tree.
-->

In [ ]:
import os, sys
from pathlib import Path
REPO = Path(os.environ.get('LONG_AUDIO_REPO', Path.cwd())).resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from __future__ import annotations
import json
from collections import defaultdict
from pathlib import Path
import ipywidgets as W
from IPython.display import HTML, display

REPO = REPO
RUNS_DIR = REPO / "runs"
MANIFESTS = {
    "ego4d":   REPO / "datasets" / "ego4d" / "annotated_manifest.json",
    "egolife": REPO / "datasets" / "egolife" / "manifest.json",
}
DATASETS = list(MANIFESTS.keys())

def discover_run_roots() -> list[tuple[str, Path]]:
    """Return [(label, eval_descriptions_dir), ...] for every runs/... subtree
    that has an eval_descriptions/{ego4d,egolife} directory. Includes
    desc-v4-*, thinking-*, and every ablation-windowsize/<N>min/."""
    out: list[tuple[str, Path]] = []
    if not RUNS_DIR.is_dir(): return out
    for top in sorted(RUNS_DIR.iterdir()):
        if not top.is_dir() or top.name.startswith(".trash"): continue
        # Direct eval_descriptions/ under top? (desc-v4-*, thinking-*)
        if (top / "eval_descriptions").is_dir():
            out.append((top.name, top / "eval_descriptions"))
        # Nested one level (ablation-windowsize/<N>min/eval_descriptions/)
        for child in sorted(top.iterdir()) if top.is_dir() else []:
            if child.is_dir() and (child / "eval_descriptions").is_dir():
                out.append((f"{top.name}/{child.name}", child / "eval_descriptions"))
    return out

RUN_ROOTS = discover_run_roots()
print(f"discovered {len(RUN_ROOTS)} run roots with eval outputs:")
for lbl, _ in RUN_ROOTS: print(f"  • {lbl}")


In [ ]:
# Manifests + GT p1 (anchor) + GT p2 (only ego4d, used as a pseudo-model)
_manifest_cache: dict[str, dict] = {}
def load_manifest(ds: str) -> dict:
    if ds not in _manifest_cache:
        _manifest_cache[ds] = json.loads(MANIFESTS[ds].read_text())
    return _manifest_cache[ds]

def gt_p1_segments_for(ds: str, uid: str) -> list[dict]:
    """Row anchor. One entry per GT p1 slice (ego4d) or summary (egolife)."""
    m = load_manifest(ds)
    v = next(x for x in m["videos"] if x["uid"] == uid)
    if ds == "ego4d":
        raw = v["passes"]["1"]["slices"]
        return [{"start": float(s["start"]), "end": float(s["end"]),
                 "description": s.get("summary") or s.get("raw_summary") or "",
                 "facts": list(s.get("facts") or [])}
                for s in raw]
    raw = v["passes"]["1"]["summaries"]
    return [{"start": float(s["start"]), "end": float(s["end"]),
             "description": s.get("text_en") or "",
             "facts": list(s.get("facts") or [])}
            for s in raw]

def gt_p2_segments_for(ds: str, uid: str) -> list[dict]:
    """ego4d only — p2 slices in the same shape as pred segments."""
    if ds != "ego4d":
        return []
    m = load_manifest(ds)
    v = next(x for x in m["videos"] if x["uid"] == uid)
    return [{"start": float(s["start"]), "end": float(s["end"]),
             "description": s.get("summary") or s.get("raw_summary") or "",
             "facts": list(s.get("facts") or []),
             "label": ""}
            for s in v["passes"]["2"]["slices"]]


In [ ]:
# Manifests + GT p1 (anchor) + GT p2 (only ego4d, used as a pseudo-model)
_manifest_cache: dict[str, dict] = {}
def load_manifest(ds: str) -> dict:
    if ds not in _manifest_cache:
        _manifest_cache[ds] = json.loads(MANIFESTS[ds].read_text())
    return _manifest_cache[ds]

def gt_p1_segments_for(ds: str, uid: str) -> list[dict]:
    """Row anchor. One entry per GT p1 slice (ego4d) or summary (egolife)."""
    m = load_manifest(ds)
    v = next(x for x in m["videos"] if x["uid"] == uid)
    if ds == "ego4d":
        raw = v["passes"]["1"]["slices"]
        return [{"start": float(s["start"]), "end": float(s["end"]),
                 "description": s.get("summary") or s.get("raw_summary") or "",
                 "facts": list(s.get("facts") or [])}
                for s in raw]
    raw = v["passes"]["1"]["summaries"]
    return [{"start": float(s["start"]), "end": float(s["end"]),
             "description": s.get("text_en") or "",
             "facts": list(s.get("facts") or [])}
            for s in raw]

def gt_p2_segments_for(ds: str, uid: str) -> list[dict]:
    """ego4d only — p2 slices in the same shape as pred segments."""
    if ds != "ego4d":
        return []
    m = load_manifest(ds)
    v = next(x for x in m["videos"] if x["uid"] == uid)
    return [{"start": float(s["start"]), "end": float(s["end"]),
             "description": s.get("summary") or s.get("raw_summary") or "",
             "facts": list(s.get("facts") or []),
             "label": ""}
            for s in v["passes"]["2"]["slices"]]


In [ ]:
# HTML renderers
def _vx(ok: bool) -> str:
    return f'<span style="color:{"#0a0" if ok else "#c00"};font-weight:600">{"\u2713" if ok else "\u2717"}</span>'

def _fmt_time(s: float, e: float) -> str:
    def fmt(x: float) -> str: return f"{int(x//60):d}:{int(x%60):02d}"
    return f"{fmt(s)}\u2013{fmt(e)}"

def _overlaps(a_s: float, a_e: float, b_s: float, b_e: float) -> bool:
    return a_s < b_e and b_s < a_e

_TH = 'style="border:1px solid #ccc;padding:6px;background:#f6f6f6;text-align:left"'
_TD = 'style="border:1px solid #ccc;padding:6px;vertical-align:top"'

def _seg_facts_block(seg: dict, prec_lookup: dict | None) -> str:
    facts = seg.get("facts") or []
    if not facts:
        return '<div style="color:#888">no atoms</div>'
    items = []
    key_window = (seg["start"], seg["end"])
    for fact in facts:
        prefix = ""
        if prec_lookup is not None:
            row = prec_lookup.get((fact, key_window))
            if row is not None:
                prefix = f'{_vx(row["is_covered"])} '
        items.append(f'<li>{prefix}{fact}</li>')
    return f'<ol style="margin:4px 0 0 0;padding-left:18px">{"".join(items)}</ol>'

def _seg_raw_block(seg: dict) -> str:
    desc = seg.get("description") or ""
    return f'<div style="margin-top:4px">{desc}</div>' if desc else '<div style="color:#888">no description</div>'

def _render_source_cell(segs: list[dict], prec_lookup: dict | None, mode: str) -> str:
    if not segs:
        return '<span style="color:#888">\u2014</span>'
    parts = []
    for seg in segs:
        parts.append(f'<div style="color:#666;font-size:11px;margin-top:6px">{_fmt_time(seg["start"], seg["end"])}</div>')
        if mode == "facts":
            parts.append(_seg_facts_block(seg, prec_lookup))
        else:
            parts.append(_seg_raw_block(seg))
    return "".join(parts)

def render_table(ds: str, uid: str, sb_lbl: str, sc_lbl: str, gt_p1: list[dict],
                 data_b: dict, data_c: dict, mode: str) -> str:
    html: list[str] = [f'<h3 style="margin-bottom:6px">{ds} \u00b7 {uid}  <span style="font-weight:normal;color:#666;font-size:12px">({mode})</span></h3>']
    html.append('<table style="border-collapse:collapse;font-size:12px;width:100%;table-layout:fixed">')
    html.append('<colgroup><col style="width:33%"><col style="width:33%"><col style="width:34%"></colgroup>')
    html.append('<thead><tr>')
    html.append(f'<th {_TH}>GT p1 (anchor)</th><th {_TH}>{sb_lbl}</th><th {_TH}>{sc_lbl}</th>')
    html.append('</tr></thead><tbody>')
    for slc in gt_p1:
        s, e = slc["start"], slc["end"]
        overlapping_b = [seg for seg in data_b["segments"] if _overlaps(s, e, seg["start"], seg["end"])]
        overlapping_c = [seg for seg in data_c["segments"] if _overlaps(s, e, seg["start"], seg["end"])]
        # GT p1 (col 1): no verdicts (it IS the truth)
        col1 = _render_source_cell([slc], None, mode)
        col2 = _render_source_cell(overlapping_b, data_b.get("precision") if mode == "facts" else None, mode)
        col3 = _render_source_cell(overlapping_c, data_c.get("precision") if mode == "facts" else None, mode)
        html.append(f'<tr><td {_TD}>{col1}</td><td {_TD}>{col2}</td><td {_TD}>{col3}</td></tr>')
    html.append('</tbody></table>')
    return "".join(html)


In [ ]:
# Source discovery + loaders — flattened across all discovered run roots.
# A "source" is either the sentinel GT_P2_SRC or a (eval_root, cell_name) pair.
# Public option keys are display strings; a registry maps back to the pair.
GT_P2_SRC = "__gt_p2__"

def _root_short(label: str) -> str:
    if label.startswith("desc-v4-"): return "desc-v4"
    if label.startswith("ablation-windowsize/"): return "abw-" + label.split("/", 1)[1]
    if label.startswith("thinking-"): return "think-" + label.removeprefix("thinking-")[:8]
    return label

def build_source_registry(ds: str) -> dict[str, tuple[Path | None, str]]:
    """{display_label -> (eval_root, cell_name) or (None, GT_P2_SRC)}.
    GT_P2_SRC (ego4d only) uses eval_root=None because human_baseline is
    manifest-derived and root-agnostic — we search all roots for any trace."""
    reg: dict[str, tuple[Path | None, str]] = {}
    if ds == "ego4d":
        reg[GT_P2_SRC] = (None, GT_P2_SRC)
    for lbl, er in RUN_ROOTS:
        ds_dir = er / ds
        if not ds_dir.is_dir(): continue
        short = _root_short(lbl)
        for p in sorted(ds_dir.iterdir()):
            if p.is_dir() and (p / "cells" / p.name).is_dir():
                reg[f"{short} / {p.name}"] = (er, p.name)
    return reg

def _find_human_baseline_trace_any(ds: str, uid: str) -> Path | None:
    """Search across all discovered roots for a human_baseline trace for this uid."""
    for _lbl, er in RUN_ROOTS:
        ds_dir = er / ds
        if not ds_dir.is_dir(): continue
        for cell_dir in sorted(ds_dir.iterdir()):
            p = cell_dir / "human_baseline" / uid / "entailment_trace.jsonl"
            if p.is_file():
                return p
    return None

def load_source(spec: tuple[Path | None, str], ds: str, uid: str) -> dict | None:
    """spec = (eval_root, cell_name) or (None, GT_P2_SRC).
    Returns {segments, precision: {(fact,(s,e)) -> row}} or None."""
    eval_root, source = spec
    if source == GT_P2_SRC:
        segs = gt_p2_segments_for(ds, uid)
        hb = _find_human_baseline_trace_any(ds, uid)
        if hb is None: return None
        lookup: dict = {}
        with hb.open() as f:
            for line in f:
                r = json.loads(line)
                if r["side"] == "human_summary_precision_A_p1_gt_p2_pred":
                    lookup[(r["hypothesis"], tuple(r["hypothesis_window"]))] = r
        return {"segments": segs, "precision": lookup}
    if eval_root is None:
        return None
    base = eval_root / ds / source / "cells" / source / uid
    pf = base / "predicted_facts.json"
    tr = base / "entailment_trace.jsonl"
    if not pf.exists() or not tr.exists(): return None
    raw_segs = json.loads(pf.read_text())["segments"]
    segs = [{"start": float(s["start"]), "end": float(s["end"]),
             "description": s.get("description") or "",
             "facts": list(s.get("facts") or []),
             "label": s.get("label", "") or ""}
            for s in raw_segs]
    lookup = {}
    with tr.open() as f:
        for line in f:
            r = json.loads(line)
            if r["side"] == "summary_precision_p1":
                lookup[(r["hypothesis"], tuple(r["hypothesis_window"]))] = r
    return {"segments": segs, "precision": lookup}

def source_uids(spec: tuple[Path | None, str], ds: str) -> set[str]:
    eval_root, source = spec
    if source == GT_P2_SRC:
        # Union of all roots' human_baseline uids (all equal in practice)
        uids: set[str] = set()
        for _lbl, er in RUN_ROOTS:
            ds_dir = er / ds
            if not ds_dir.is_dir(): continue
            for cell_dir in ds_dir.iterdir():
                hb = cell_dir / "human_baseline"
                if hb.is_dir():
                    for u in hb.iterdir():
                        if (u / "entailment_trace.jsonl").is_file():
                            uids.add(u.name)
                    break
        return uids
    if eval_root is None: return set()
    cr = eval_root / ds / source / "cells" / source
    if not cr.is_dir(): return set()
    return {p.name for p in cr.iterdir()
            if p.is_dir() and (p / "predicted_facts.json").is_file()}

def common_uids(specs: list[tuple[Path | None, str]], ds: str) -> list[str]:
    if not specs: return []
    inter = None
    for spec in specs:
        u = source_uids(spec, ds)
        inter = u if inter is None else inter & u
    return sorted(inter or [])


In [ ]:
# Widgets + main
_dataset = W.Dropdown(options=DATASETS, description="dataset:")
_src_b   = W.Dropdown(description="col 2:", layout=W.Layout(width="720px"))
_src_c   = W.Dropdown(description="col 3:", layout=W.Layout(width="720px"))
_uid     = W.Dropdown(description="uid:",   layout=W.Layout(width="720px"))
_mode    = W.ToggleButtons(options=[("Atomic facts", "facts"), ("Raw descriptions", "raw")],
                           value="facts", description="show:")
_status  = W.HTML()
_btn     = W.Button(description="Render", button_style="primary")
_out     = W.Output()

# Registry rebuilt on dataset change.
_registry: dict[str, tuple] = {}

def _refresh_sources(*_):
    global _registry
    _registry = build_source_registry(_dataset.value)
    labels = list(_registry.keys())
    _src_b.options = labels
    _src_c.options = labels
    if len(labels) >= 2 and _src_b.value == _src_c.value:
        _src_c.value = labels[1] if labels[0] == _src_b.value else labels[0]
    _refresh_uids()

def _refresh_uids(*_):
    picks = [_registry.get(v) for v in (_src_b.value, _src_c.value) if v]
    picks = [p for p in picks if p is not None]
    uids = common_uids(picks, _dataset.value) if len(picks) == 2 else []
    _uid.options = uids
    _status.value = f'<span style="color:#666">{len(uids)} uids with both selected sources evaluated</span>'

def _render(*_):
    with _out:
        _out.clear_output()
        ds, sb_lbl, sc_lbl, uid, mode = _dataset.value, _src_b.value, _src_c.value, _uid.value, _mode.value
        sb, sc = _registry.get(sb_lbl), _registry.get(sc_lbl)
        if not (ds and sb and sc and uid):
            print("select dataset + both sources + uid"); return
        gt_p1 = gt_p1_segments_for(ds, uid)
        db = load_source(sb, ds, uid)
        dc = load_source(sc, ds, uid)
        if db is None or dc is None:
            missing = [n for n, x in ((sb_lbl, db), (sc_lbl, dc)) if x is None]
            print(f"missing eval outputs for: {missing}"); return
        display(HTML(render_table(ds, uid, sb_lbl, sc_lbl, gt_p1, db, dc, mode)))

_dataset.observe(_refresh_sources, names="value")
_src_b.observe(_refresh_uids, names="value")
_src_c.observe(_refresh_uids, names="value")
_mode.observe(_render, names="value")
_btn.on_click(_render)

_refresh_sources()
display(W.VBox([_dataset, _src_b, _src_c, _uid, _mode, _status, _btn, _out]))
